# SQ-GAN CPU data benchmark
No training, no GPU use, no production changes. 800 images per measured pass.

In [ ]:
DATASET_SOURCE='from __future__ import annotations\n\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\n\nclass DIV2KWithPseudoLabels(Dataset):\n    """Paired RGB and 19-class Cityscapes-ID pseudo-labels at author\'s 2:1 geometry."""\n    def __init__(self, image_root: str, label_root: str, size=(256, 512), train: bool = True):\n        self.image_root = Path(image_root)\n        self.label_root = Path(label_root)\n        if isinstance(size, int):\n            self.height = self.width = int(size)\n        else:\n            self.height, self.width = map(int, size)\n        self.train = bool(train)\n        exts = {".png", ".jpg", ".jpeg"}\n        images = sorted(p for p in self.image_root.rglob("*") if p.suffix.lower() in exts)\n        if not images:\n            raise FileNotFoundError(f"No images under {self.image_root}")\n        self.pairs = []\n        missing = []\n        for image in images:\n            mask = self.label_root / f"{image.stem}.png"\n            if mask.is_file():\n                self.pairs.append((image, mask))\n            else:\n                missing.append(str(mask))\n        if missing:\n            raise FileNotFoundError(f"Missing {len(missing)} pseudo masks; first: {missing[0]}")\n        self.image_norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def _resize_cover(self, image, mask):\n        scale = max(self.width / image.width, self.height / image.height)\n        new_size = (max(self.width, round(image.width * scale)), max(self.height, round(image.height * scale)))\n        image = image.resize(new_size, Image.Resampling.BICUBIC)\n        mask = mask.resize(new_size, Image.Resampling.NEAREST)\n        return image, mask\n\n    def __getitem__(self, idx):\n        image_path, mask_path = self.pairs[idx]\n        image = Image.open(image_path).convert("RGB")\n        mask = Image.open(mask_path).convert("L")\n        image, mask = self._resize_cover(image, mask)\n        if self.train:\n            x = int(np.random.randint(0, image.width - self.width + 1))\n            y = int(np.random.randint(0, image.height - self.height + 1))\n        else:\n            x = (image.width - self.width) // 2\n            y = (image.height - self.height) // 2\n        box = (x, y, x + self.width, y + self.height)\n        image, mask = image.crop(box), mask.crop(box)\n        if self.train and np.random.rand() < 0.5:\n            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n        image_tensor = self.image_norm(transforms.ToTensor()(image))\n        ids = torch.from_numpy(np.asarray(mask, dtype=np.uint8).copy()).long()\n        if torch.any((ids != 255) & (ids > 18)):\n            raise ValueError(f"Expected Cityscapes IDs [0,18] or 255: {mask_path}")\n        ids[ids == 255] = 19\n        semantic = torch.nn.functional.one_hot(ids, num_classes=20).permute(2, 0, 1).float()\n        label_ids = ids.clone()\n        label_ids[label_ids == 19] = 0\n        label = label_ids.unsqueeze(0).float()\n        return {"image": image_tensor, "label": label, "semantic": semantic, "filename": image_path.name}\n'
exec(DATASET_SOURCE,globals())

import json, time, hashlib
from torch.utils.data import DataLoader
from PIL import Image
torch.set_num_threads(1)
OUT=Path('/kaggle/working/sqgan_cpu_data_benchmark'); OUT.mkdir(exist_ok=True)
prior=next(Path('/kaggle/input').rglob('sqgan_div2k_channel_results'))
images=Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images')
labels=prior/'pseudo_labels/div2k_train'
original=DIV2KWithPseudoLabels(str(images),str(labels),size=(256,512),train=True)
assert len(original)==800
cache=Path('/tmp/sqgan_cpu_cache'); cache.mkdir(exist_ok=True)
mapping={}; equality=[]
t0=time.perf_counter()
for i,(image_path,mask_path) in enumerate(original.pairs):
    with Image.open(image_path) as im, Image.open(mask_path) as ma:
        image,mask=original._resize_cover(im.convert('RGB'),ma.convert('L'))
    cp,mp=cache/f'{i}_rgb.png',cache/f'{i}_label.png'
    image.save(cp,compress_level=1); mask.save(mp,compress_level=1)
    mapping[str(image_path)]=(cp,mp)
cache_build_seconds=time.perf_counter()-t0
cached=DIV2KWithPseudoLabels(str(images),str(labels),size=(256,512),train=True)
cached.pairs=[mapping[str(p[0])] for p in original.pairs]
for idx in range(0,800,40):
    np.random.seed(20261020+idx); a=original[idx]
    np.random.seed(20261020+idx); b=cached[idx]
    same=all(torch.equal(a[k],b[k]) for k in ('image','label','semantic'))
    assert same,idx
    equality.append({'index':idx,'image_label_semantic_exactly_equal':same})
(OUT/'cache_equivalence.json').write_text(json.dumps(equality,indent=2))
reports=[]
for name,dataset,workers,pin in [('original_workers0',original,0,False),
                                ('cached_workers0',cached,0,False),
                                ('cached_workers2',cached,2,False)]:
    gen=torch.Generator().manual_seed(20261020)
    loader=DataLoader(dataset,batch_size=1,shuffle=False,num_workers=workers,
                      pin_memory=pin,generator=gen,persistent_workers=workers>0)
    seen=[]; t0=time.perf_counter(); first100=None
    for batch in loader:
        seen.extend(batch['filename'])
        assert batch['image'].shape==(1,3,256,512)
        if len(seen)==100: first100=time.perf_counter()-t0
    seconds=time.perf_counter()-t0
    assert len(seen)==len(set(seen))==800
    report={'variant':name,'cpu_only':True,'images_processed':800,'unique_images':800,
            'batch_size':1,'workers':workers,'full_pass_seconds':seconds,
            'first100_seconds':first100,'remaining700_seconds':seconds-first100,
            'cache_build_seconds':cache_build_seconds if 'cached' in name else 0,
            'gpu_training_speedup_measured':False,'production_changed':False}
    reports.append(report)
    (OUT/'data_loading_report.json').write_text(json.dumps(reports,indent=2))
    print('CPU DATA RESULT',json.dumps(report),flush=True)
print('CPU DATA BENCHMARK COMPLETE',flush=True)
